# 03 Lesion Detection

Demonstra e valida a Etapa 3 (F3): detecção de lesões dentro da folha, usando ranges HSV para amarelos e marrons.

Fluxo: `bgr_to_hsv` -> `segment_leaf` (F2, para obter `leaf_mask`) -> `detect_lesions` (F3) -> `draw_lesion_contours`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))

from plant_disease.config import SELECTED_DATA_DIR
from plant_disease.utils.io import load_batch
from plant_disease.utils.color import bgr_to_hsv, bgr_to_rgb
from plant_disease.stage2_leaf_seg import segment_leaf, extract_largest_contour
from plant_disease.stage3_lesion import (
    LesionDetectionConfig,
    detect_lesions,
    draw_lesion_contours,
)

%matplotlib inline
plt.rcParams["figure.figsize"] = (16, 5)

print("✓ Imports OK")

In [ ]:
# Categorias saudáveis vs doentes para demonstrar F3
categories = [
    "tomato_healthy",
    "tomato_early_blight",
    "tomato_late_blight",
    "potato_healthy",
    "potato_early_blight",
    "potato_late_blight",
]

demo_images = []
for category in categories:
    cat_dir = SELECTED_DATA_DIR / category
    if cat_dir.exists():
        batch = load_batch(str(cat_dir), recursive=False)
        for path, img in batch[:3]:
            demo_images.append((path, img, category))

print(f"✓ Loaded {len(demo_images)} imagens de {len(categories)} categorias")

In [ ]:
# Parâmetros de segmentação da folha (F2, já validados em 02_leaf_seg.ipynb)
LEAF_HSV_PARAMS = dict(h_min=35, h_max=85, s_min=30, s_max=255, v_min=40, v_max=255)

lesion_config = LesionDetectionConfig()


def processar_imagem(img_bgr):
    """Executa F2 (segmentação da folha) + F3 (detecção de lesões) em uma imagem."""
    hsv = bgr_to_hsv(img_bgr)

    leaf_result = segment_leaf(hsv, aplicar_morph=True, **LEAF_HSV_PARAMS)
    if leaf_result is None:
        return None
    raw_leaf_mask, _ = leaf_result

    contour_result = extract_largest_contour(raw_leaf_mask)
    if contour_result is None:
        return None
    leaf_mask, leaf_area = contour_result

    lesion_result = detect_lesions(hsv, leaf_mask, config=lesion_config)
    if lesion_result is None:
        return None
    lesion_mask, contours = lesion_result

    lesion_px = int((lesion_mask > 0).sum())
    pct = (lesion_px / leaf_area * 100) if leaf_area > 0 else 0.0

    drawn = draw_lesion_contours(img_bgr, contours)

    return {
        "leaf_mask": leaf_mask,
        "lesion_mask": lesion_mask,
        "contours": contours,
        "drawn": drawn,
        "leaf_area": leaf_area,
        "lesion_px": lesion_px,
        "pct": pct,
    }


print("✓ Helper processar_imagem() definido")

In [ ]:
def mostrar_resultado(path, img_bgr, category):
    r = processar_imagem(img_bgr)
    if r is None:
        print(f"Falha ao processar: {path}")
        return

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

    axes[0].imshow(bgr_to_rgb(img_bgr))
    axes[0].set_title("Original")

    axes[1].imshow(r["leaf_mask"], cmap="gray")
    axes[1].set_title("Máscara da Folha (F2)")

    axes[2].imshow(r["lesion_mask"], cmap="gray")
    axes[2].set_title(f"Máscara de Lesão (F3)\n{r['lesion_px']} px")

    axes[3].imshow(bgr_to_rgb(r["drawn"]))
    axes[3].set_title(f"Contornos ({len(r['contours'])})")

    for ax in axes:
        ax.axis("off")

    plt.suptitle(
        f"{category} — {Path(path).name} — {r['pct']:.1f}% da folha",
        fontsize=12,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.show()


# Demonstra em algumas imagens de cada categoria (saudáveis + doentes)
for path, img_bgr, category in demo_images[:12]:
    mostrar_resultado(path, img_bgr, category)

## Validação de critérios (F3)

1. A máscara de lesão deve estar sempre contida na máscara da folha (`lesion & ~leaf == 0`).
2. Folhas saudáveis devem apresentar percentual de lesão próximo de zero.
3. Folhas doentes devem apresentar contornos coerentes com as regiões manchadas.

In [ ]:
import cv2

linhas = []
falhas_contencao = 0

for path, img_bgr, category in demo_images:
    r = processar_imagem(img_bgr)
    if r is None:
        continue

    fora_da_folha = cv2.countNonZero(
        cv2.bitwise_and(r["lesion_mask"], cv2.bitwise_not(r["leaf_mask"]))
    )
    if fora_da_folha > 0:
        falhas_contencao += 1

    linhas.append(
        {
            "image": Path(path).name,
            "category": category,
            "leaf_px": r["leaf_area"],
            "lesion_px": r["lesion_px"],
            "pct": round(r["pct"], 2),
            "n_contours": len(r["contours"]),
            "lesao_fora_da_folha_px": fora_da_folha,
        }
    )

import pandas as pd

df = pd.DataFrame(linhas)
display(df)

print(f"\nImagens com lesão fora da folha: {falhas_contencao}/{len(df)} (deve ser 0)")

saudaveis = df[df["category"].str.contains("healthy")]
if len(saudaveis) > 0:
    print(f"Percentual médio de lesão em folhas saudáveis: {saudaveis['pct'].mean():.2f}% (esperado próximo de 0)")

## Observações e próximos ajustes

- Se folhas saudáveis apresentarem percentual de lesão elevado, revisar os ranges de amarelo em `LesionDetectionConfig` (nervuras/reflexos podem cair no range).
- Se lesões visíveis não forem capturadas, avaliar alargar levemente os ranges de marrom ou reduzir `min_contour_area`.
- Qualquer ajuste feito aqui deve ser refletido em `stage3_lesion.py` (config), não hardcoded no notebook.